Data Exploration

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
from PIL import Image
from collections import Counter

data_dir = Path("images")
size = []
bad_images = []

for folder in data_dir.iterdir():
    if folder.is_dir():
        images = list(folder.glob("*.jpg"))
        print("folder name:", folder, "number of images:", len(images))
        for image_path in images:
            try:
                img = Image.open(image_path)
                size.append(img.size)
                img.verify()
            except Exception as e:
                bad_images.append(image_path)
                print("bad image",image_path, "|Error", e)

        plt.figure(figsize=(8,3))
        plt.suptitle(folder.name)
        plt_images = images[:3]
        for i, img_path in enumerate(plt_images):
            img = Image.open(img_path)
            plt.subplot(1, 3, i+1)
            plt.imshow(img)
            plt.axis("off")
        plt.show()

size_counts = Counter(size)
print("Total bad images", len(bad_images),"\n")
print("sizes of images:", size_counts.most_common(10),"\n" )




Check GreyScale Images vs RGB images

In [ ]:
image_modes = []
grey_scale_images = []
for folder in data_dir.iterdir():
    
    image_paths = list(folder.glob("*.jpg"))
    
    for image_path in image_paths:
        img = Image.open(image_path)
        image_modes.append(img.mode)
        if img.mode == 'L':
            grey_scale_images.append(image_path)
    
image_modes_num = Counter(image_modes)
print(image_modes_num)
print(grey_scale_images)

Split Images into different folders for traing, test and validate

In [ ]:
import shutil
import random

output_dir = Path("images_split")
train = 0.7
val = 0.1
test = 0.2
random.seed(42)
for class_folder in data_dir.iterdir():
    if class_folder.is_dir():
        class_name = class_folder.name
        images_org = list(class_folder.glob("*.jpg"))
        random.shuffle(images_org)

        total_images = len(images_org)

        train_num = int(train * total_images)
        val_num = int(val * total_images)

        train_images = images_org[:train_num]
        val_images = images_org[train_num :train_num + val_num]
        test_images = images_org[train_num + val_num: ]

        for split_name , split_images in [
            ("train", train_images ),
            ("val",  val_images),
            ("test",test_images )]:

            split_dir = output_dir/ split_name/ class_folder.name
            split_dir.mkdir(parents = True, exist_ok = True)

            for image_path in split_images:
                shutil.copy(image_path, split_dir/image_path.name)

        print(class_name, class_folder)
        print("train length:", len(train_images))
        print("val length:", len(val_images))
        print("test length:", len(test_images),  "\n")

Check the split image dataset

In [ ]:
split_dir_path  = Path("images_split")

for split in ["test", "train", "val"]:
     for class_folder in (split_dir_path/split).iterdir():
         class_images_sum = sum(1 for img in class_folder.glob("*.jpg"))
         print(split, class_folder,class_images_sum )



In [ ]:
import tensorflow as tf

## Create Training split Data

In [ ]:
batch_siz = 32
img_size = (150,150)
train_data = tf.keras.utils.image_dataset_from_directory(
            "images_split/train",
            image_size = img_size,
            batch_size = batch_siz,
            color_mode = "rgb",
            shuffle = True
            )

print(train_data.class_names, "\n")
print(train_data)
print(train_data.element_spec)

Create test Data Split

In [ ]:
test_data = tf.keras.utils.image_dataset_from_directory(
            "images_split/test",
            image_size= img_size,
            color_mode = "rgb",
            batch_size = batch_siz     
            )
test_data.class_names

print(test_data.element_spec)

Create Validation test split

In [ ]:
val_data = tf.keras.utils.image_dataset_from_directory(
            "images_split/val",
            image_size= img_size,
            color_mode = "rgb",
            batch_size = batch_siz     
            )
val_data.class_names
print(val_data.element_spec)

In [ ]:
for images, labels in train_data.take(1):
    print("Image shape:", images.shape)
    print("label shape:", labels.shape)
    print("lables for first 10:", labels[:10])
    print("Min pixel value:", images.numpy().min())
    print("Max pixel value:", images.numpy().max())


Check images in train set in batch one

In [ ]:
class_name = train_data.class_names
for images, labels in train_data.take(1):
    for i, image in enumerate(images):
        plt.subplot(4,8, i+1)
        plt.imshow(image.numpy().astype("uint8"))
        plt.title(class_name[labels[i]])
        plt.axis("off")
    plt.show()

Check the cat, dog and panda category per splt data set for test, train and validate

In [ ]:
for data_set_name, data_set in [("train",train_data), ("test", test_data), ("val",val_data)]:
    label_counter = Counter()
    for images, label in data_set:
        label_counter.update(label.numpy())
    print(data_set_name, label_counter)

## Number of batches created  per split 

In [ ]:
print("train_batches:", len(train_data))
print("test_batches:", len(test_data))
print("val_batches:", len(val_data))

In [ ]:
from tensorflow.keras import layers, models
from tensorflow.keras.models import Model
from tensorflow.keras.layers import  Dense, Input, Conv2D, Conv1D, MaxPooling2D, GlobalMaxPooling2D, Rescaling, Dropout

## BaseLine model with two Conv2D layers eaxh with (3,3) filter and 2 Maxpooling Flattening is acheived by GlobalMaxPooling2D

In [ ]:
image_input = Input(shape=(150,150,3), name = "image_input")
rescale  = Rescaling(1./255)(image_input)
cnn1 = Conv2D(32,(3,3), activation="relu")(rescale)
pooling_layer = MaxPooling2D((2,2))(cnn1)

cnn2 = Conv2D(32,(3,3), activation="relu")(pooling_layer)
pooling_layer2 = MaxPooling2D((2,2))(cnn2)

flatten = GlobalMaxPooling2D()(pooling_layer2)

connected_layer = Dense(128, activation="relu")(flatten)
output_layer = Dense(3, activation="softmax")(connected_layer)

In [ ]:
model = Model(
    inputs = image_input,
    outputs = output_layer
)

In [ ]:
model.compile(
    optimizer = "adam",
    loss = "sparse_categorical_crossentropy",
    metrics = ["accuracy"]
)

In [ ]:
model.summary()

In [ ]:
history = model.fit(
    train_data,
    validation_data =  val_data,
    epochs = 30
)

In [ ]:
test_loss, test_accuracy = model.evaluate(test_data)
print("Baseline :test loss:",test_loss,"test_accuracy", test_accuracy)

In [ ]:
plt.plot(history.history["accuracy"], label="train_accuracy")
plt.plot(history.history["val_accuracy"], label="val_accuracy")
plt.legend()
plt.show()

In [ ]:
plt.plot(history.history["val_loss"], label="val_loss")
plt.plot(history.history["loss"], label="train_loss")
plt.legend()
plt.show()

## Confususion Matrix

In [ ]:
import numpy as np
y_actual =[]
y_predict = []
for images, labels in test_data:
    predictions = model.predict(images)
    #print(predictions)
    prediction_act = np.argmax(predictions, axis=1)
    #print(prediction_act)
    y_actual.extend(labels.numpy())
    y_predict.extend(prediction_act)

y_actual = np.array(y_actual)
y_predict = np.array(y_predict)

confusion_matrix = tf.math.confusion_matrix(y_actual, y_predict)
print(confusion_matrix.numpy())

### Confusion Matrix Display

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
class_names = ["cats", "dogs", "panda"]

disp = ConfusionMatrixDisplay(
    confusion_matrix.numpy(),
    display_labels=class_name
)
disp.plot()
plt.show()

## Model with Augmentation

In [ ]:
from tensorflow.keras.layers import RandomFlip, RandomRotation, RandomZoom, RandomContrast

data_augmentation =  tf.keras.Sequential([
    RandomFlip("horizontal"),
    RandomRotation(0.08),
    RandomZoom(.10),
    RandomContrast(.10)
], name = "data_augmentation")


In [ ]:
image_input = Input(shape=(150,150,3), name = "image_input")

augmented_data = data_augmentation(image_input)

rescale  = Rescaling(1./255)(augmented_data)
cnn1 = Conv2D(32,(3,3), activation="relu")(rescale)
pooling_layer = MaxPooling2D((2,2))(cnn1)

cnn2 = Conv2D(32,(3,3), activation="relu")(pooling_layer)
pooling_layer2 = MaxPooling2D((2,2))(cnn2)

flatten = GlobalMaxPooling2D()(pooling_layer2)

connected_layer = Dense(128, activation="relu")(flatten)
output_layer = Dense(3, activation="softmax")(connected_layer)

In [ ]:
model2 = Model(
    inputs = image_input,
    outputs = output_layer
)

model2.compile(
    optimizer = "adam",
    loss = "sparse_categorical_crossentropy",
    metrics = ["accuracy"]
)
model2.summary()

In [ ]:
history = model2.fit(
    train_data,
    validation_data =  val_data,
    epochs = 30
)

In [ ]:
test_loss, test_accuracy = model2.evaluate(test_data)
print("test loss Augmented data line:",test_loss,"test_accuracy", test_accuracy)

## VGG Style model + Data Augmentation 

In [ ]:

image_input = Input(shape=(150,150,3), name = "image_input")
## Data Augmentation
#augmented_data = data_augmentation(image_input)
##Rescaling
x  = Rescaling(1./255)(image_input)
##VVG block 1
x = Conv2D(32,(3,3), activation="relu", padding = "same")(x)
x = Conv2D(32,(3,3), activation="relu",padding = "same")(x)
x = MaxPooling2D((2,2))(x)

##VVG block 2
x = Conv2D(64,(3,3), activation="relu", padding = "same")(x)
x = Conv2D(64,(3,3), activation="relu",padding = "same")(x)
x = MaxPooling2D((2,2))(x)

# ##VVG block 3 
## Din not train well with this block so commented 
# x = Conv2D(128,(3,3), activation="relu", padding = "same")(x)
# x = Conv2D(128,(3,3), activation="relu",padding = "same")(x)
# x = MaxPooling2D((2,2))(x)
# x = Dropout(0.25)(x)


flatten = GlobalMaxPooling2D()(x)

x = Dense(128, activation="relu")(flatten)
output_layer = Dense(3, activation="softmax")(x)

In [ ]:
model_vgg = Model(
    inputs = image_input,
    outputs = output_layer
)

model_vgg.compile(
    optimizer = "adam",
    loss = "sparse_categorical_crossentropy",
    metrics = ["accuracy"]
)
model_vgg.summary()
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor = "val_accuracy",
    patience = 4,
    mode = max,
    restore_best_weights = True
)

In [ ]:
history = model_vgg.fit(
    train_data,
    validation_data =  val_data,
    epochs = 25,
    callbacks =[early_stop]
)

In [ ]:
test_loss, test_accuracy = model_vgg.evaluate(test_data)
print("test loss:",test_loss,"test_accuracy", test_accuracy)

## CNN with RSMProp as optimizer

In [ ]:
image_input = Input(shape=(150,150,3), name = "image_input")
rescale  = Rescaling(1./255)(image_input)
cnn1 = Conv2D(32,(3,3), activation="relu")(rescale)
pooling_layer = MaxPooling2D((2,2))(cnn1)

cnn2 = Conv2D(32,(3,3), activation="relu")(pooling_layer)
pooling_layer2 = MaxPooling2D((2,2))(cnn2)

flatten = GlobalMaxPooling2D()(pooling_layer2)

connected_layer = Dense(128, activation="relu")(flatten)
output_layer = Dense(3, activation="softmax")(connected_layer)

model_rsm = Model(
    inputs = image_input,
    outputs = output_layer
)

model_rsm.compile(
    optimizer = "rmsprop",
    loss = "sparse_categorical_crossentropy",
    metrics = ["accuracy"]
)

history = model_rsm.fit(
    train_data,
    validation_data =  val_data,
    epochs = 30
)

In [ ]:
test_loss, test_accuracy = model_rsm.evaluate(test_data)
print("test rsmprop loss:",test_loss,"test_accuracy", test_accuracy)

Four models were compared: a baseline CNN using Adam, the same CNN with data augmentation, a VGG-style CNN with early stopping, and the baseline CNN using RMSprop. The baseline CNN with Adam achieved the best overall test performance, with a test accuracy of 73.83% and the lowest test loss of 0.5796. The CNN with RMSprop performed similarly with 72.83% test accuracy and a loss of 0.5909, while the model with data augmentation also achieved 72.83% accuracy but had a higher test loss of 0.6764. The VGG-style CNN produced the lowest test accuracy of 68.33% with a test loss of 0.6955. These results show that increasing model complexity or adding augmentation did not improve performance for this dataset and configuration. Therefore, the simpler baseline CNN with Adam was the most effective model among the approaches tested.